# money_v1: raw ZIP to clean classification source

**Purpose:** keep all readable, correctly annotated, one-banknote representatives from this source after the frozen bbox gate and natural-context crop.

This notebook only cleans one source. It does not balance classes, create final train/dev/test splits, or train a model.


## 1. Imports and global configuration



In [1]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime
from PIL import Image
from IPython.display import display
import pandas as pd
import yaml
import math
import re
import io
import hashlib

In [2]:
DATASET_ID = "money_v1"
RAW_ZIP = Path.home() / "Downloads" / "Money.v1i.yolov5pytorch.zip"

OUTPUT_ROOT = Path(r"C:\CS231_RAW_NOTEBOOK_RUNS\money_v1_simple")
RUN_DIR = OUTPUT_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = RUN_DIR

DATA_OUT = RUN_ROOT / "MONEY_V1_CLEAN"

TARGET_CLASSES = ["10000", "20000", "50000", "100000", "200000", "500000"]
BBOX_MIN_SHORT_SIDE = 120
CONTEXT_RATIO = 0.20

assert RAW_ZIP.is_file(), f"Không tìm thấy ZIP: {RAW_ZIP}"

RUN_ROOT.mkdir(parents=True, exist_ok=True)

archive = ZipFile(RAW_ZIP)
members = set(archive.namelist())

print("Raw ZIP :", RAW_ZIP)
print("Output  :", DATA_OUT)

Raw ZIP : C:\Users\Nguyen\Downloads\Money.v1i.yolov5pytorch.zip
Output  : C:\CS231_RAW_NOTEBOOK_RUNS\money_v1_simple\20261026_004048\MONEY_V1_CLEAN


## 2. Open the ZIP, read data.yaml, and inventory the export



In [3]:
config_member = next(
    name for name in sorted(members)
    if name.endswith("/data.yaml") or name == "data.yaml"
)

config = yaml.safe_load(
    archive.read(config_member).decode("utf-8-sig")
)

class_names = config["names"]

def normalize_denomination(class_name):
    text = str(class_name).strip().lower()

    if text.endswith("k"):
        number = int(re.sub(r"\D", "", text))
        return str(number * 1000)

    return re.sub(r"\D", "", text)

class_to_vnd = {
    class_id: normalize_denomination(class_name)
    for class_id, class_name in enumerate(class_names)
}

class_table = pd.DataFrame([
    {
        "class_id": class_id,
        "raw_class_name": class_names[class_id],
        "denomination": class_to_vnd[class_id],
        "used": class_to_vnd[class_id] in TARGET_CLASSES,
    }
    for class_id in range(len(class_names))
])

display(class_table)

,class_id,raw_class_name,denomination,used
0,0,100k,100000,True
1,1,10k,10000,True
2,2,1k,1000,False
3,3,200k,200000,True
4,4,20k,20000,True
5,5,2k,2000,False
6,6,500k,500000,True
7,7,50k,50000,True
8,8,5k,5000,False


## 3. Helper functions and source metadata



In [4]:
image_members = [
    name for name in sorted(members)
    if "/images/" in name
    and Path(name).suffix.lower() in {".jpg", ".jpeg", ".png"}
]

def label_for(image_member):
    label_name = image_member.replace("/images/", "/labels/")
    return label_name.rsplit(".", 1)[0] + ".txt"

def source_name(member):
    name = Path(member).stem
    name = re.sub(r"\.rf\.[0-9a-f]+$", "", name, flags=re.I)
    name = re.sub(r"_(jpg|jpeg|png)$", "", name, flags=re.I)
    return name

def original_split(member):
    return member.split("/images/")[0].split("/")[-1]

def read_one_box(label_member):
    if label_member not in members:
        return None

    lines = [
        line.strip()
        for line in archive.read(label_member).decode("utf-8-sig").splitlines()
        if line.strip()
    ]

    if len(lines) != 1:
        return None

    parts = lines[0].split()
    if len(parts) != 5:
        return None

    try:
        class_id = int(parts[0])
        cx, cy, bw, bh = map(float, parts[1:])
    except ValueError:
        return None

    if class_id not in class_to_vnd:
        return None

    if not (
        math.isfinite(cx)
        and math.isfinite(cy)
        and math.isfinite(bw)
        and math.isfinite(bh)
        and 0 <= cx <= 1
        and 0 <= cy <= 1
        and 0 < bw <= 1
        and 0 < bh <= 1
    ):
        return None

    return {
        "raw_label": class_to_vnd[class_id],
        "cx": cx,
        "cy": cy,
        "bw": bw,
        "bh": bh,
    }

print("Số file ảnh export trong ZIP:", len(image_members))

Số file ảnh export trong ZIP: 12020


## 4. Build one valid representative per source



In [5]:
records = []

for member in image_members:
    label_member = label_for(member)
    box = read_one_box(label_member)

    try:
        raw_bytes = archive.read(member)
        with Image.open(io.BytesIO(raw_bytes)) as image:
            image.load()
            width, height = image.size
        readable = True
    except Exception:
        width, height = 0, 0
        readable = False

    valid = (
        readable
        and box is not None
        and box["raw_label"] in TARGET_CLASSES
    )

    row = {
        "member": member,
        "annotation_member": label_member,
        "source_name": source_name(member),
        "original_split": original_split(member),
        "width": width,
        "height": height,
        "source_image_sha256": hashlib.sha256(raw_bytes).hexdigest() if readable else "",
        "valid": valid,
    }

    if box is None:
        row.update({
            "raw_label": "",
            "cx": 0.0,
            "cy": 0.0,
            "bw": 0.0,
            "bh": 0.0,
        })
    else:
        row.update(box)

    records.append(row)

all_images = pd.DataFrame(records)

all_images = all_images.sort_values(
    ["source_name", "valid", "member"],
    ascending=[True, False, True]
)

representatives = all_images.drop_duplicates("source_name", keep="first").copy()

pool = (
    representatives[representatives["valid"]]
    .copy()
    .reset_index(drop=True)
)

print("Exported image files :", len(all_images))
print("Unique source groups :", len(representatives))
print("Valid target sources :", len(pool))
print("Discarded/out-of-scope:", len(representatives) - len(pool))

display(
    pool.groupby("raw_label")
        .size()
        .reindex(TARGET_CLASSES, fill_value=0)
        .to_frame("valid_sources")
)

Exported image files : 12020
Unique source groups : 12019
Valid target sources : 7310
Discarded/out-of-scope: 4709


,valid_sources
raw_label,
10000,1241
20000,1039
50000,1030
100000,1221
200000,1666
500000,1113


## 5. Verified corrections/removals and capture provenance

The correction/removal maps are empty for this source. Filename-derived capture dates are kept only as provenance.


In [6]:
CORRECT_LABEL_BY_SHA256 = {}
REMOVE_BY_SHA256 = {}
verified_relabels = 0
verified_removals = 0
print("Verified relabels:", verified_relabels)
print("Verified removals:", verified_removals)

def capture_info(member):
    match = re.search(
        r"IMG_(\d{8})_(\d{6})",
        Path(member).stem,
        flags=re.I
    )

    if match:
        return match.group(1), match.group(2)

    return "", ""

capture_dates = []
capture_times = []

for member in pool["member"]:
    date, time = capture_info(member)
    capture_dates.append(date)
    capture_times.append(time)

pool["capture_date"] = capture_dates
pool["capture_time"] = capture_times

print("Có capture_date:", (pool["capture_date"] != "").sum(), "/", len(pool))
print(
    "Số ngày chụp khác nhau:",
    pool.loc[pool["capture_date"] != "", "capture_date"].nunique()
)

Verified relabels: 0
Verified removals: 0


Có capture_date: 7300 / 7310
Số ngày chụp khác nhau: 32


## 6. Apply the frozen bbox threshold



In [7]:
def pixel_bbox(row):
    width = int(row["width"])
    height = int(row["height"])

    left = row["cx"] - row["bw"] / 2
    top = row["cy"] - row["bh"] / 2
    right = row["cx"] + row["bw"] / 2
    bottom = row["cy"] + row["bh"] / 2

    x1 = max(0, math.floor(left * width))
    y1 = max(0, math.floor(top * height))
    x2 = min(width, math.ceil(right * width))
    y2 = min(height, math.ceil(bottom * height))

    return x1, y1, x2, y2

bbox_rows = []

for row in pool.to_dict("records"):
    x1, y1, x2, y2 = pixel_bbox(row)

    bbox_rows.append({
        "bbox_x1": x1,
        "bbox_y1": y1,
        "bbox_x2": x2,
        "bbox_y2": y2,
        "bbox_short_side_px": min(x2 - x1, y2 - y1),
    })

bbox_df = pd.DataFrame(bbox_rows)

pool = pd.concat(
    [pool.reset_index(drop=True), bbox_df],
    axis=1
)

before_scale_filter = len(pool)

below_120 = pool[
    pool["bbox_short_side_px"] < BBOX_MIN_SHORT_SIDE
].copy()

pool = pool[
    pool["bbox_short_side_px"] >= BBOX_MIN_SHORT_SIDE
].copy().reset_index(drop=True)

print("Trước bbox filter :", before_scale_filter)
print("Sau bbox >= BBOX_MIN_SHORT_SIDEpx :", len(pool))
print("Bị loại vì quá nhỏ:", len(below_120))

display(
    pool.groupby("raw_label")
        .size()
        .reindex(TARGET_CLASSES, fill_value=0)
        .to_frame("after_bbox_filter")
)

Trước bbox filter : 7310
Sau bbox >= BBOX_MIN_SHORT_SIDEpx : 6378
Bị loại vì quá nhỏ: 932


,after_bbox_filter
raw_label,
10000,1038
20000,912
50000,883
100000,1077
200000,1519
500000,949


## 7. Crop with 20% context



In [8]:
def context_box(row):
    x1 = int(row["bbox_x1"])
    y1 = int(row["bbox_y1"])
    x2 = int(row["bbox_x2"])
    y2 = int(row["bbox_y2"])

    width = int(row["width"])
    height = int(row["height"])

    box_width = x2 - x1
    box_height = y2 - y1

    crop_x1 = max(0, round(x1 - CONTEXT_RATIO * box_width))
    crop_y1 = max(0, round(y1 - CONTEXT_RATIO * box_height))
    crop_x2 = min(width, round(x2 + CONTEXT_RATIO * box_width))
    crop_y2 = min(height, round(y2 + CONTEXT_RATIO * box_height))

    return crop_x1, crop_y1, crop_x2, crop_y2

## 8. Export kept samples and write the manifest



In [9]:
manifest_rows = []

for row in pool.sort_values(["raw_label", "source_name"]).to_dict("records"):
    raw_bytes = archive.read(row["member"])

    with Image.open(io.BytesIO(raw_bytes)) as image:
        image = image.convert("RGB")
        crop_box = context_box(row)
        crop = image.crop(crop_box)

    safe_source = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        row["source_name"]
    )

    relative_path = (
        f"{row['raw_label']}/"
        f"MONEY_V1_{safe_source}.png"
    )

    target = DATA_OUT / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)

    crop.save(
        target,
        format="PNG",
        optimize=False
    )

    manifest_rows.append({
        "relative_path": relative_path,
        "dataset_id": "money_v1",
        "source_name": row["source_name"],
        "capture_group": row["source_name"],
        "source_image_sha256": row["source_image_sha256"],
        "final_crop_sha256": hashlib.sha256(target.read_bytes()).hexdigest(),
        "source_archive_member": row["member"],
        "annotation_member": row["annotation_member"],
        "original_split": row["original_split"],
        "raw_label": row["raw_label"],
        "final_label": row["raw_label"],
        "capture_date": row["capture_date"],
        "capture_time": row["capture_time"],
        "source_width": row["width"],
        "source_height": row["height"],
        "bbox_x1": row["bbox_x1"],
        "bbox_y1": row["bbox_y1"],
        "bbox_x2": row["bbox_x2"],
        "bbox_y2": row["bbox_y2"],
        "bbox_short_side_px": row["bbox_short_side_px"],
        "crop_x1": crop_box[0],
        "crop_y1": crop_box[1],
        "crop_x2": crop_box[2],
        "crop_y2": crop_box[3],
        "crop_width": crop.width,
        "crop_height": crop.height,
    })

result = pd.DataFrame(manifest_rows)

result.to_csv(
    DATA_OUT / "manifest.csv",
    index=False
)

print("Tổng ảnh output:", len(result))

display(
    result.groupby("final_label")
          .size()
          .reindex(TARGET_CLASSES, fill_value=0)
          .to_frame("images")
)

Tổng ảnh output: 6378


,images
final_label,
10000,1038
20000,912
50000,883
100000,1077
200000,1519
500000,949


## 9. Final source-run audit



In [10]:
assert result["relative_path"].is_unique
assert result["source_name"].is_unique
assert set(result["final_label"]).issubset(set(TARGET_CLASSES))
assert (result["bbox_short_side_px"].astype(int) >= BBOX_MIN_SHORT_SIDE).all()
assert len(list(DATA_OUT.rglob("*.png"))) == len(result)

class_counts = (
    result.groupby("final_label")
          .size()
          .reindex(TARGET_CLASSES, fill_value=0)
)

summary_lines = [
    "MONEY V1 SIMPLE PIPELINE",
    "========================",
    f"Raw exported files: {len(all_images)}",
    f"Unique source groups: {len(representatives)}",
    f"Valid target sources before bbox filter: {before_scale_filter}",
    f"Removed by bbox < BBOX_MIN_SHORT_SIDEpx: {len(below_120)}",
    f"Kept after bbox >= BBOX_MIN_SHORT_SIDEpx: {len(result)}",
    "",
    "Class counts:",
]

for label, count in class_counts.items():
    summary_lines.append(f"{label}: {count}")

summary_text = "\n".join(summary_lines)

(RUN_ROOT / "RUN_SUMMARY.txt").write_text(
    summary_text,
    encoding="utf-8"
)

print(summary_text)

MONEY V1 SIMPLE PIPELINE
Raw exported files: 12020
Unique source groups: 12019
Valid target sources before bbox filter: 7310
Removed by bbox < BBOX_MIN_SHORT_SIDEpx: 932
Kept after bbox >= BBOX_MIN_SHORT_SIDEpx: 6378

Class counts:
10000: 1038
20000: 912
50000: 883
100000: 1077
200000: 1519
500000: 949


In [11]:
# Tất cả các con số dưới đây được tính lại từ manifest và PNG vừa xuất.
manifest = result if "result" in globals() else manifest
label_col = "final_label" if "final_label" in manifest.columns else "denomination"
png_files = list(DATA_OUT.rglob("*.png"))
unreadable_png = 0
for path in png_files:
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception:
        unreadable_png += 1

if "before_scale_filter" in globals():
    valid_before_bbox = int(before_scale_filter)
else:
    valid_before_bbox = len(pool)
removed_by_bbox = valid_before_bbox - len(manifest)
class_counts = manifest.groupby(label_col).size().reindex(TARGET_CLASSES, fill_value=0).to_dict()
invalid_labels = sorted(set(manifest[label_col].astype(str)) - set(TARGET_CLASSES))
path_label_mismatch = sum(
    str(row["relative_path"]).replace("\\", "/").split("/")[0] != str(row[label_col])
    for _, row in manifest.iterrows()
)
min_bbox = int(manifest["bbox_short_side_px"].astype(int).min()) if len(manifest) else 0
duplicate_source = int(manifest["source_name"].duplicated().sum())
duplicate_path = int(manifest["relative_path"].duplicated().sum())
relabeled = len(corrections_found) if "corrections_found" in globals() else int(globals().get("verified_relabels", 0))
removed_by_review = int(globals().get("verified_removals", 0))
checks_pass = (
    len(manifest) == len(png_files)
    and unreadable_png == 0
    and duplicate_source == 0
    and duplicate_path == 0
    and invalid_labels == []
    and path_label_mismatch == 0
    and min_bbox >= BBOX_MIN_SHORT_SIDE
)
source_run_status = "PASS" if checks_pass else "HOLD"

summary_lines = [
    f"DATASET_ID = {DATASET_ID}",
    f"RAW_IMAGES = {len(all_images)}",
    f"SOURCE_GROUPS = {len(representatives)}",
    f"VALID_BEFORE_BBOX = {valid_before_bbox}",
    f"VERIFIED_RELABELS = {relabeled}",
    f"VERIFIED_REMOVALS = {removed_by_review}",
    f"BBOX_LT_120_REMOVED = {removed_by_bbox}",
    f"FINAL_KEPT = {len(manifest)}",
    f"FINAL_CLASS_COUNTS = {class_counts}",
    f"MANIFEST_ROWS = {len(manifest)}",
    f"PNG_COUNT = {len(png_files)}",
    f"DUPLICATE_SOURCE_NAME = {duplicate_source}",
    f"DUPLICATE_RELATIVE_PATH = {duplicate_path}",
    f"INVALID_FINAL_LABELS = {invalid_labels}",
    f"PATH_LABEL_MISMATCH = {path_label_mismatch}",
    f"UNREADABLE_EXPORTED_PNG = {unreadable_png}",
    f"MIN_BBOX_SHORT_SIDE_PX = {min_bbox}",
    f"OUTPUT_ROOT = {DATA_OUT}",
    f"SOURCE_RUN_STATUS = {source_run_status}",
]
summary_text = "\n".join(summary_lines)
(RUN_ROOT / "RUN_SUMMARY.txt").write_text(summary_text, encoding="utf-8")
print(summary_text)
assert source_run_status == "PASS", "Source output audit did not pass; preserve run outputs for review."


DATASET_ID = money_v1
RAW_IMAGES = 12020
SOURCE_GROUPS = 12019
VALID_BEFORE_BBOX = 7310
VERIFIED_RELABELS = 0
VERIFIED_REMOVALS = 0
BBOX_LT_120_REMOVED = 932
FINAL_KEPT = 6378
FINAL_CLASS_COUNTS = {'10000': 1038, '20000': 912, '50000': 883, '100000': 1077, '200000': 1519, '500000': 949}
MANIFEST_ROWS = 6378
PNG_COUNT = 6378
DUPLICATE_SOURCE_NAME = 0
DUPLICATE_RELATIVE_PATH = 0
INVALID_FINAL_LABELS = []
PATH_LABEL_MISMATCH = 0
UNREADABLE_EXPORTED_PNG = 0
MIN_BBOX_SHORT_SIDE_PX = 120
OUTPUT_ROOT = C:\CS231_RAW_NOTEBOOK_RUNS\money_v1_simple\20261026_004048\MONEY_V1_CLEAN
SOURCE_RUN_STATUS = PASS
